# Sinh bài — Ollama (Qwen3.5:2b, thinking / non-thinking)

Sinh `output` cho từng case qua Ollama, **không chạy eval**.

**Input:** `dataset/fnb_dataset_test.json`  
**Output:** `results/generations/qwen3.5_2b_[thinking|nothinking]_TIMESTAMP.json`

Toggle `GENERATION_THINKING = True/False` để so sánh thinking vs non-thinking — hai file output riêng biệt giúp đưa vào eval độc lập.

In [50]:
# %pip install -q requests pandas python-dotenv

In [51]:
import os
import re
from pathlib import Path
from datetime import datetime, timezone, timedelta
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
load_dotenv(_dotenv_path, override=True, encoding="utf-8")
ROOT = Path(_dotenv_path).resolve().parent if _dotenv_path else Path.cwd().resolve()

# ===== Hyperparameters =====
DATASET_REL        = "dataset/fnb_dataset_test.json"
OUTPUT_REL         = "results/generations"

# Ollama settings
OLLAMA_HOST        = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").rstrip("/").removesuffix("/v1")
LOCAL_MODEL_ID     = (os.getenv("OLLAMA_MODEL_ID") or "qwen3.5:2b").strip()

# *** Đổi True/False ở đây để chọn chế độ thinking ***
GENERATION_THINKING = False

MAX_RETRIES        = 3

DATASET_PATH = ROOT / DATASET_REL
OUTPUT_DIR   = ROOT / OUTPUT_REL

mode_tag   = "thinking" if GENERATION_THINKING else "nothinking"
VN_TZ      = timezone(timedelta(hours=7))
RUN_ID     = datetime.now(VN_TZ).strftime("%d-%m-%Y_%H-%M")
safe_model = re.sub(r"[^A-Za-z0-9._-]+", "_", LOCAL_MODEL_ID).strip("_") or "model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# output_path = OUTPUT_DIR / f"qwen3.5_9b_nothinking_21-05-2026_19-36.json"
output_path = OUTPUT_DIR / f"{safe_model}_{mode_tag}_{RUN_ID}.json"

print(f"Model : {LOCAL_MODEL_ID} @ {OLLAMA_HOST}")
print(f"Mode  : {GENERATION_THINKING} → tag: {mode_tag}")
print(f"Input : {DATASET_PATH}")
print(f"Output: {output_path}")


Model : qwen3.5-4b-facebook-content @ http://192.168.92.26:11434
Mode  : False → tag: nothinking
Input : D:\Github\mcs-train-content-model\dataset\fnb_dataset_test.json
Output: D:\Github\mcs-train-content-model\results\generations\qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05.json


In [52]:
import json
import sys
import time
import requests

_OPTIONS_THINKING = {
    "temperature":       0.7,
    "presence_penalty":  0.0,
    "frequency_penalty": 0.0, # Không phạt khi mô hình suy nghĩ quá đà
    "top_p":              0.9,
}
_OPTIONS_NO_THINKING = {
    "temperature": 0.4,
}

_THINK_TAG_RE = re.compile(r"<think>.*?</think>\s*", re.DOTALL)


def stream_ollama_chat(messages, think=True, model=None, verbose=True):
    """
    Stream chat từ Ollama native API (/api/chat).
    verbose=False: tắt in ra màn hình, dùng cho batch generation.
    Trả về dict:
        thinking            : str   — nội dung thinking của LLM
        answer              : str   — nội dung trả lời (đã strip <think> tags nếu có)
        time_thinking       : float — giây LLM dành cho thinking (0.0 nếu think=False)
        time_generation     : float — giây sinh answer (bao gồm network latency khi think=False)
        time_total          : float — tổng thời gian
        tokens_thinking     : int   — số token thinking (đếm chunk streaming)
        tokens_answer       : int   — số token answer (eval_count từ Ollama done chunk)
        tokens_total        : int   — tokens_thinking + tokens_answer
    """
    model   = model or LOCAL_MODEL_ID
    options = _OPTIONS_THINKING if think else _OPTIONS_NO_THINKING

    payload = {
        "model":    model,
        "messages": messages,
        "stream":   True,
        "think":    think,
        "options":  options,
    }

    GREY  = "\033[90m"
    BOLD  = "\033[1m"
    CYAN  = "\033[36m"
    RESET = "\033[0m"

    thinking_buf    = ""
    answer_buf      = ""
    in_thinking     = False
    answer_start_t  = None
    tokens_thinking = 0
    tokens_answer   = 0
    start           = time.perf_counter()

    with requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload, stream=True, timeout=300,
    ) as resp:
        resp.raise_for_status()
        for raw_line in resp.iter_lines():
            if not raw_line:
                continue
            chunk = json.loads(raw_line.decode("utf-8"))
            msg   = chunk.get("message", {})

            thinking_tok = msg.get("thinking", "")
            if thinking_tok:
                if verbose and not in_thinking:
                    in_thinking = True
                    sys.stdout.write(f"{GREY}[Thinking]\n")
                if verbose:
                    sys.stdout.write(thinking_tok)
                thinking_buf    += thinking_tok
                tokens_thinking += 1

            content_tok = msg.get("content", "")
            if content_tok:
                if answer_start_t is None:
                    answer_start_t = time.perf_counter()
                    if verbose:
                        sep = f"{RESET}\n\n" if in_thinking else ""
                        thinking_elapsed = answer_start_t - start
                        label = f"thinking: {thinking_elapsed:.1f}s" if thinking_buf else f"TTFT: {thinking_elapsed:.1f}s"
                        sys.stdout.write(f"{sep}{BOLD}[Trả lời]  ({label}){RESET}\n")
                        in_thinking = False
                if verbose:
                    sys.stdout.write(content_tok)
                answer_buf += content_tok

            if chunk.get("done"):
                end           = time.perf_counter()
                total         = end - start
                tokens_answer = chunk.get("eval_count", 0)
                if answer_start_t is None:
                    answer_start_t = end

                if thinking_buf:
                    time_thinking   = answer_start_t - start
                    time_generation = end - answer_start_t
                else:
                    time_thinking   = 0.0
                    time_generation = total

                tokens_total = tokens_thinking + tokens_answer
                ns  = chunk.get("eval_duration", 0)
                tps = tokens_answer / (ns / 1e9) if ns > 0 else 0
                if verbose:
                    sys.stdout.write(
                        f"\n\n{CYAN}── total: {total:.1f}s | {tokens_total} tok"
                        f" | thinking: {time_thinking:.1f}s ({tokens_thinking} tok)"
                        f" | gen: {time_generation:.1f}s ({tokens_answer} tok)"
                        f" | {tps:.1f} tok/s ──{RESET}\n"
                    )
                    sys.stdout.flush()
                break

    # Strip <think>...</think> khỏi answer phòng trường hợp Ollama không tách riêng
    # (xảy ra với custom Modelfile chưa được Ollama nhận diện thinking protocol)
    answer_buf = _THINK_TAG_RE.sub("", answer_buf).strip()

    return {
        "thinking":         thinking_buf,
        "answer":           answer_buf,
        "time_thinking":    round(time_thinking, 3),
        "time_generation":  round(time_generation, 3),
        "time_total":       round(total, 3),
        "tokens_thinking":  tokens_thinking,
        "tokens_answer":    tokens_answer,
        "tokens_total":     tokens_total,
    }


print("stream_ollama_chat: sẵn sàng.")

stream_ollama_chat: sẵn sàng.


In [53]:
# Verify kết nối — demo 1 câu ngắn
stream_ollama_chat([{"role": "user", "content": "Xin chào"}], think=GENERATION_THINKING, verbose=True)

[Trả lời]  (TTFT: 3.6s)
<think>

</think>

Nếu bạn đang tìm một thương hiệu thực phẩm sạch có quy trình kiểm soát chặt chẽ, minh bạch và dễ tiếp cận cho gia đình, VINA FOOD là lựa chọn đáng cân nhắc. Với hệ thống sản xuất theo tiêu chuẩn GHP-GMP và quy trình kiểm soát chất lượng từ khâu nguyên liệu đến thành phẩm, VINA FOOD cam kết mang lại bữa ăn an toàn, sạch và tiện lợi cho người tiêu dùng Việt.

Thương hiệu tập trung vào các nhóm sản phẩm thực phẩm tươi sống, chế biến nhẹ và đóng gói tiện lợi, phù hợp cho nhu cầu ăn uống hàng ngày của hộ gia đình, văn phòng và các đơn vị catering. VINA FOOD không chỉ bán sản phẩm — họ xây dựng niềm tin thông qua minh bạch nguồn gốc, quy trình sản xuất rõ ràng và dịch vụ chăm sóc khách hàng chuyên nghiệp.

Nếu bạn muốn tìm một đối tác thực phẩm sạch có trách nhiệm với chất lượng và an toàn, hãy inbox VINA FOOD ngay hôm nay để nhận tư vấn miễn phí hoặc đặt thử sản phẩm đầu tiên.

── total: 12.2s | 219 tok | thinking: 0.0s (0 tok) | gen: 12.2s (219 to

{'thinking': '',
 'answer': 'Nếu bạn đang tìm một thương hiệu thực phẩm sạch có quy trình kiểm soát chặt chẽ, minh bạch và dễ tiếp cận cho gia đình, VINA FOOD là lựa chọn đáng cân nhắc. Với hệ thống sản xuất theo tiêu chuẩn GHP-GMP và quy trình kiểm soát chất lượng từ khâu nguyên liệu đến thành phẩm, VINA FOOD cam kết mang lại bữa ăn an toàn, sạch và tiện lợi cho người tiêu dùng Việt.\n\nThương hiệu tập trung vào các nhóm sản phẩm thực phẩm tươi sống, chế biến nhẹ và đóng gói tiện lợi, phù hợp cho nhu cầu ăn uống hàng ngày của hộ gia đình, văn phòng và các đơn vị catering. VINA FOOD không chỉ bán sản phẩm — họ xây dựng niềm tin thông qua minh bạch nguồn gốc, quy trình sản xuất rõ ràng và dịch vụ chăm sóc khách hàng chuyên nghiệp.\n\nNếu bạn muốn tìm một đối tác thực phẩm sạch có trách nhiệm với chất lượng và an toàn, hãy inbox VINA FOOD ngay hôm nay để nhận tư vấn miễn phí hoặc đặt thử sản phẩm đầu tiên.',
 'time_thinking': 0.0,
 'time_generation': 12.177,
 'time_total': 12.177,
 'toke

In [54]:
stream_ollama_chat([{"role": "user", "content": "Xin chào"}], think=True, verbose=True)

HTTPError: 400 Client Error: Bad Request for url: http://192.168.92.26:11434/api/chat

In [55]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

if not isinstance(dataset, list):
    raise ValueError("Dataset phải là mảng JSON.")

missing_ids = [i for i, r in enumerate(dataset) if "id" not in r]
if missing_ids:
    raise ValueError(f"Thiếu field 'id' ở {len(missing_ids)} record: indices {missing_ids[:5]}")

print(f"Loaded {len(dataset)} records từ {DATASET_PATH.name}")
print(f"ID range: {dataset[0]['id']} → {dataset[-1]['id']}")


Loaded 100 records từ fnb_dataset_test.json
ID range: 1 → 100


In [56]:
# Demo 1 bài mẫu để xác nhận chất lượng trước khi chạy batch
_demo = dataset[13]
_user_msg = (
    f"Viết bài marketing Markdown từ tiêu đề và mồi:\n\n"
    f"Tiêu đề: {_demo['title']}\n\n"
    f"Mồi:\n{_demo['seed']}"
)

print("=" * 60)
print("[SYSTEM PROMPT]")
print(_demo["instruction"])
print()
print("[USER PROMPT]")
print(_user_msg)
print("=" * 60)

_demo_result = stream_ollama_chat(
    messages=[
        {"role": "system", "content": _demo["instruction"]},
        {"role": "user",   "content": _user_msg},
    ],
    think=GENERATION_THINKING,
)
print("\nThinking chars:", len(_demo_result["thinking"]))

[SYSTEM PROMPT]
Bạn là copywriter marketing người Việt, giọng director marketing F&B dày dạn, viết Facebook Ads native, rõ offer, ngắn gọn và có CTA chốt đơn. Bám sát seed, ưu tiên hook bán hàng, không hype rỗng, không bịa claim ngoài dữ liệu đã cho.

[USER PROMPT]
Viết bài marketing Markdown từ tiêu đề và mồi:

Tiêu đề: Viết caption Facebook Ads cho Seoul Fire BBQ — buffet lẩu nướng 199K, hook đi nhóm càng lời.

Mồi:
Thương hiệu: Seoul Fire BBQ — chuỗi lẩu nướng, 8 điểm bán Hà Nội
Sản phẩm: Buffet Seoul 199K — 40+ món thịt bò, heo, gà, panchan
Giá: 199.000đ (gốc 269.000đ, -26%), tặng Pepsi refill
Điều kiện: 17h-22h T2-T5, đặt inbox FB, áp dụng 4 người trở lên
Đối tượng: 22-35 tuổi, dân văn phòng, Cầu Giấy, Đống Đa, Thanh Xuân
Kênh: Facebook Page, Messenger, Zalo OA
KPI: CTR ≥ 2,8%, CPL ≤ 18.000đ, ROAS ≥ 4
[Trả lời]  (TTFT: 0.3s)
<think>

</think>

Đi nhóm văn phòng mà vẫn muốn ăn vui, no và không bị “đi chợ” là gì?

Buffet Seoul 199K tại Seoul Fire BBQ đang chốt deal cho khung 17h-22h

In [57]:
def build_user_prompt(title: str, seed: str) -> str:
    return (
        f"Viết bài marketing Markdown từ tiêu đề và mồi:\n\n"
        f"Tiêu đề: {title}\n\n"
        f"Mồi:\n{seed}"
    )

def _save(path, recs):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(recs, f, ensure_ascii=False, indent=2)

# --- Resume: nạp bản ghi đã có nếu file output tồn tại ---
records   = []
seen_ids  = set()
if output_path.exists():
    with open(output_path, "r", encoding="utf-8") as f:
        records = json.load(f)
    seen_ids = {r["id"] for r in records if r.get("id")}
    print(f"Resume: tìm thấy {len(records)} bản ghi đã có trong {output_path.name}")

_t0 = time.perf_counter()

for idx, row in enumerate(dataset, start=1):
    row_id = row["id"]

    if row_id in seen_ids:
        print(f"[{idx}/{len(dataset)}] Bỏ qua (đã có id={row_id}): {row['title'][:55]!r}")
        continue

    elapsed_total = time.perf_counter() - _t0
    done  = len(records)
    empty = sum(1 for r in records if not r["output"])
    print(
        f"\n[{idx}/{len(dataset)}] Đang xử lý: {row['title'][:60]!r}"
        f" | xong: {done} ({empty} rỗng) | tổng: {elapsed_total:.1f}s",
        flush=True,
    )

    for attempt in range(MAX_RETRIES):
        try:
            r = stream_ollama_chat(
                messages=[
                    {"role": "system", "content": row["instruction"]},
                    {"role": "user",   "content": build_user_prompt(row["title"], row["seed"])},
                ],
                think=GENERATION_THINKING,
                verbose=False,
            )
            rec = {
                "id":               row_id,
                "instruction":      row["instruction"],
                "title":            row["title"],
                "seed":             row["seed"],
                "output":           r["answer"].strip(),
                "thinking":         r["thinking"],
                "time_thinking":    r["time_thinking"],
                "time_generation":  r["time_generation"],
                "time_total":       r["time_total"],
                "tokens_thinking":  r["tokens_thinking"],
                "tokens_answer":    r["tokens_answer"],
                "tokens_total":     r["tokens_total"],
            }
            records.append(rec)
            seen_ids.add(row_id)
            _save(output_path, records)          # lưu ngay sau mỗi bài
            print(
                f"  ✓ id={row_id}"
                f" | thinking: {r['time_thinking']:.1f}s ({r['tokens_thinking']} tok)"
                f" | gen: {r['time_generation']:.1f}s ({r['tokens_answer']} tok)"
                f" | total: {r['time_total']:.1f}s ({r['tokens_total']} tok)",
                flush=True,
            )
            break
        except Exception as e:
            print(f"  Lỗi lần {attempt + 1}/{MAX_RETRIES}: {e!r}", flush=True)
            if attempt + 1 == MAX_RETRIES:
                rec = {
                    "id":               row_id,
                    "instruction":      row["instruction"],
                    "title":            row["title"],
                    "seed":             row["seed"],
                    "output":           "",
                    "thinking":         "",
                    "time_thinking":    0.0,
                    "time_generation":  0.0,
                    "time_total":       0.0,
                    "tokens_thinking":  0,
                    "tokens_answer":    0,
                    "tokens_total":     0,
                }
                records.append(rec)
                seen_ids.add(row_id)
                _save(output_path, records)      # lưu ngay kể cả khi lỗi



[1/100] Đang xử lý: 'Viết caption Facebook cho landing page ưu đãi của Bếp Quê OC' | xong: 0 (0 rỗng) | tổng: 0.0s
  ✓ id=1 | thinking: 0.0s (0 tok) | gen: 7.3s (182 tok) | total: 7.3s (182 tok)

[2/100] Đang xử lý: 'Viết caption Facebook cho chiến dịch Trung Thu của Matcha Mâ' | xong: 1 (0 rỗng) | tổng: 7.3s
  ✓ id=2 | thinking: 0.0s (0 tok) | gen: 19.3s (487 tok) | total: 19.3s (487 tok)

[3/100] Đang xử lý: 'Viết caption Facebook cho activation tại điểm bán của Cầu Tr' | xong: 2 (0 rỗng) | tổng: 26.7s
  ✓ id=3 | thinking: 0.0s (0 tok) | gen: 9.4s (235 tok) | total: 9.4s (235 tok)

[4/100] Đang xử lý: 'Viết post Facebook báo cáo hiệu quả chiến dịch combo trưa củ' | xong: 3 (0 rỗng) | tổng: 36.1s
  ✓ id=4 | thinking: 0.0s (0 tok) | gen: 13.2s (329 tok) | total: 13.2s (329 tok)

[5/100] Đang xử lý: "Viết caption Facebook cho Herb n' Glow Detox Tea — combo 7 n" | xong: 4 (0 rỗng) | tổng: 49.3s
  ✓ id=5 | thinking: 0.0s (0 tok) | gen: 21.9s (552 tok) | total: 21.9s (552 tok)

[6/100] Đa

In [58]:
done    = [r for r in records if r["output"]]
empty   = [r for r in records if not r["output"]]

def _avg(recs, key):
    vals = [r[key] for r in recs if key in r]
    return sum(vals) / len(vals) if vals else 0.0

print(f"Hoàn tất: {len(records)} bản ghi ({len(empty)} rỗng) đã lưu vào: {output_path}")
if done:
    print()
    print(f"{'Thống kê trung bình':─<45} (n={len(done)} bài thành công)")
    print(f"  time_thinking  : {_avg(done, 'time_thinking'):>8.2f} s   | tokens_thinking : {_avg(done, 'tokens_thinking'):>7.1f} tok")
    print(f"  time_generation: {_avg(done, 'time_generation'):>8.2f} s   | tokens_answer   : {_avg(done, 'tokens_answer'):>7.1f} tok")
    print(f"  time_total     : {_avg(done, 'time_total'):>8.2f} s   | tokens_total    : {_avg(done, 'tokens_total'):>7.1f} tok")
print()
print("Bước tiếp theo: mở evaluation/evaluate.ipynb và đặt:")
print(f'  INPUT_JSON = ROOT / "{output_path.relative_to(ROOT)}"')
print(f'  MODEL_ROLE = "trained_model"  # hoặc "open-source"')
print(f'  MODEL_ID   = "{LOCAL_MODEL_ID}_{mode_tag}"')


Hoàn tất: 100 bản ghi (0 rỗng) đã lưu vào: D:\Github\mcs-train-content-model\results\generations\qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05.json

Thống kê trung bình────────────────────────── (n=100 bài thành công)
  time_thinking  :     0.00 s   | tokens_thinking :     0.0 tok
  time_generation:    11.38 s   | tokens_answer   :   282.8 tok
  time_total     :    11.38 s   | tokens_total    :   282.8 tok

Bước tiếp theo: mở evaluation/evaluate.ipynb và đặt:
  INPUT_JSON = ROOT / "results\generations\qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05.json"
  MODEL_ROLE = "trained_model"  # hoặc "open-source"
  MODEL_ID   = "qwen3.5-4b-facebook-content_nothinking"
